# In-Class Exercise: Lasso vs. Ridge

ECON6083 | Regularization | 40 minutes

## Part I: Setup & Data (5 min)

We simulate a housing-price prediction problem with 50 features, but only 5 truly matter.
Your job: compare OLS, Ridge, and Lasso on this data.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error


### Step 1: Generate synthetic data

We create 100 observations with 50 features. Only the first 5 coefficients are non-zero.

In [ ]:
np.random.seed(42)
n_samples, n_features = 100, 50
X = np.random.randn(n_samples, n_features)

# True coefficients: only the first 5 are non-zero
true_coef = np.zeros(n_features)
true_coef[:5] = [1.5, -2, 3, 0.5, -1]
y = X @ true_coef + 0.5 * np.random.randn(n_samples)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)


### Step 2: Fit OLS

Fit a standard `LinearRegression` and compute the test MSE.

In [ ]:
ols = LinearRegression()
ols.fit(X_train, y_train)
ols_pred = ols.predict(X_test)
ols_mse = mean_squared_error(y_test, ols_pred)

print(f"OLS Test MSE: {ols_mse:.2f}")


### Step 3: Fit Ridge

Fit a `Ridge` model with `alpha=1.0` and compute the test MSE.

In [ ]:
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)
ridge_pred = ridge.predict(X_test)
ridge_mse = mean_squared_error(y_test, ridge_pred)

print(f"Ridge Test MSE: {ridge_mse:.2f}")


### Step 4: Fit Lasso

Fit a `Lasso` model with `alpha=0.1` and compute the test MSE.

In [ ]:
lasso = Lasso(alpha=0.1)
lasso.fit(X_train, y_train)
lasso_pred = lasso.predict(X_test)
lasso_mse = mean_squared_error(y_test, lasso_pred)

print(f"Lasso Test MSE: {lasso_mse:.2f}")


### Step 5: Compare

How many of the 50 coefficients did Lasso set to exactly zero?

In [ ]:
print(f"Number of non-zero coefficients in Lasso: "
      f"{np.sum(lasso.coef_ != 0)}/{n_features}")


**Discuss with your neighbor:**
- Which model had the lowest test MSE? Why?
- Does Lasso's feature count match the true DGP (5 relevant features)?

## Part II: Bias-Variance Decomposition via Monte Carlo (15 min)

We verify **Test MSE = Bias² + Variance + Irreducible Error** by
repeatedly generating training data, fitting all three models, and
measuring how predictions vary across 200 replications.

### Step 1: Setup storage and fixed test set

In [ ]:
import matplotlib.pyplot as plt

np.random.seed(42)
n_sim = 200
n_samples, n_features = 100, 50
n_test = 50
sigma = 0.5

true_coef = np.zeros(n_features)
true_coef[:5] = [1.5, -2, 3, 0.5, -1]


In [ ]:
# Fixed test set (same across all simulations)
X_test_fixed = np.random.randn(n_test, n_features)
y_true_fixed = X_test_fixed @ true_coef  # signal only

# Storage: predictions from each simulation
preds = {"OLS": [], "Ridge": [], "Lasso": []}
test_mses = {"OLS": [], "Ridge": [], "Lasso": []}


### Step 2: Run the simulation loop

In each replication, generate fresh training data and fit all three models.

In [ ]:
for r in range(n_sim):
    X_train = np.random.randn(n_samples, n_features)
    y_train = X_train @ true_coef + sigma * np.random.randn(n_samples)

    ols = LinearRegression()
    ols.fit(X_train, y_train)
    pred_ols = ols.predict(X_test_fixed)

    ridge = Ridge(alpha=1.0)
    ridge.fit(X_train, y_train)
    pred_ridge = ridge.predict(X_test_fixed)

    lasso = Lasso(alpha=0.1)
    lasso.fit(X_train, y_train)
    pred_lasso = lasso.predict(X_test_fixed)

    for name, pred in zip(
        ["OLS", "Ridge", "Lasso"],
        [pred_ols, pred_ridge, pred_lasso],
    ):
        preds[name].append(pred)
        y_test_noisy = y_true_fixed + sigma * np.random.randn(n_test)
        test_mses[name].append(np.mean((y_test_noisy - pred) ** 2))


### Step 3: Compute Bias² and Variance

In [ ]:
# Convert to arrays: shape (n_sim, n_test)
for name in preds:
    preds[name] = np.array(preds[name])
    test_mses[name] = np.array(test_mses[name])


In [ ]:
bias_sq = {}
variance = {}
mean_mse = {}

for name in ["OLS", "Ridge", "Lasso"]:
    mean_pred = preds[name].mean(axis=0)
    bias_sq[name] = np.mean((mean_pred - y_true_fixed) ** 2)
    variance[name] = np.mean(preds[name].var(axis=0))
    mean_mse[name] = test_mses[name].mean()

    print(f"{name:6s}  Bias²={bias_sq[name]:.4f}  "
          f"Var={variance[name]:.4f}  MSE={mean_mse[name]:.4f}")


### Step 4: Plot the decomposition

In [ ]:
methods = ["OLS", "Ridge", "Lasso"]
x = np.arange(len(methods))
width = 0.25

fig, ax = plt.subplots(figsize=(8, 5))

ax.bar(x - width, [bias_sq[m] for m in methods], width, label="Bias²")
ax.bar(x,         [variance[m] for m in methods], width, label="Variance")
ax.bar(x + width, [mean_mse[m] for m in methods], width, label="Test MSE")

ax.set_xticks(x)
ax.set_xticklabels(methods)
ax.set_ylabel("Value")
ax.set_title("Bias-Variance Decomposition: OLS vs Ridge vs Lasso")
ax.legend()
plt.tight_layout()
plt.show()


**Discussion:**
- Which model has the highest variance? Why?
- Which model has the highest bias? Why does regularization introduce bias?
- Does Bias² + Variance ≈ Test MSE − σ²? (σ = 0.5, so irreducible error = 0.25)
- Based on the plot, which model achieves the best bias-variance tradeoff?

## Part III: Post-Double Selection Lasso (15 min)

**Paper:** Belloni, Chernozhukov & Hansen (2014), *Review of Economic Studies*

Naively running Lasso on $Y \sim D + X$ can **drop confounders** and bias
the treatment effect. Post-double selection fixes this.

### Scenario

- $D_i$: treatment (binary), $Y_i = \alpha D_i + X_i'\beta + \epsilon_i$, true $\alpha = 2.0$
- 200 potential controls, only 10 truly matter
- Some controls predict $Y$, some predict $D$, some predict both

In [ ]:
from sklearn.linear_model import LassoCV
import statsmodels.api as sm

np.random.seed(42)
n, p = 500, 200
alpha_true = 2.0
X = np.random.randn(n, p)


In [ ]:
# Coefficients for Y equation: first 10 controls matter
beta_y = np.zeros(p)
beta_y[:10] = [1.0, 0.8, 0.6, 0.4, 0.2, -0.5, -0.3, 0.0, 0.0, 0.0]

# Coefficients for D equation: controls 5-14 matter
gamma_d = np.zeros(p)
gamma_d[5:15] = [0.6, 0.5, 0.4, 0.3, 0.2, 0.8, 0.7, 0.6, 0.5, 0.4]

# Generate treatment and outcome
D = (X @ gamma_d + np.random.randn(n) > 0).astype(float)
Y = alpha_true * D + X @ beta_y + np.random.randn(n)


In [ ]:
print(f"True treatment effect: {alpha_true}")
print(f"n = {n}, p = {p}")
print(f"Controls that affect Y: indices 0-9")
print(f"Controls that affect D: indices 5-14")
print(f"Confounders (affect both): indices 5-9")


### Step 1: Naive approaches (for comparison)

In [ ]:
# Approach 1: OLS with NO controls (omitted variable bias)
model_no_ctrl = sm.OLS(Y, sm.add_constant(D)).fit()
alpha_no_ctrl = model_no_ctrl.params[1]
print(f"No controls:   alpha_hat = {alpha_no_ctrl:.3f}  "
      f"(true = {alpha_true})")


In [ ]:
# Approach 2: Naive Lasso on Y ~ D + X (dangerous!)
DX = np.column_stack([D, X])
naive_lasso = LassoCV(cv=5).fit(DX, Y)
alpha_naive = naive_lasso.coef_[0]
print(f"Naive Lasso:   alpha_hat = {alpha_naive:.3f}  "
      f"(true = {alpha_true})")


### Step 2: Post-Double Selection (BCH procedure)

1. **Lasso of $Y$ on $X$** → select controls $S_1$
2. **Lasso of $D$ on $X$** → select controls $S_2$
3. **OLS of $Y$ on $D$ and $S_1 \cup S_2$**

In [ ]:
# Lasso Y on X → select S1
lasso_y = LassoCV(cv=5).fit(X, Y)
S1 = np.where(lasso_y.coef_ != 0)[0]
print(f"S1 (controls predicting Y): {S1}")


In [ ]:
# Lasso D on X → select S2
lasso_d = LassoCV(cv=5).fit(X, D)
S2 = np.where(lasso_d.coef_ != 0)[0]
print(f"S2 (controls predicting D): {S2}")


In [ ]:
# OLS of Y on D and union(S1, S2)
S_union = np.union1d(S1, S2)
print(f"S1 \cup S2: {S_union}")
print(f"Number of selected controls: {len(S_union)}")


Run OLS on the selected controls to get the treatment effect.

In [ ]:
X_selected = X[:, S_union]
X_final = sm.add_constant(np.column_stack([D, X_selected]))
model_pds = sm.OLS(Y, X_final).fit()
alpha_pds = model_pds.params[1]
print(f"\nPost-double selection: alpha_hat = {alpha_pds:.3f}  "
      f"(true = {alpha_true})")


### Step 3: Compare all approaches

In [ ]:
print("=" * 55)
print(f"{'Method':<25} {'alpha_hat':>10} {'Bias':>10}")
print("-" * 55)
print(f"{'True effect':<25} {alpha_true:>10.3f} {0.0:>+10.3f}")
print(f"{'No controls':<25} {alpha_no_ctrl:>10.3f} "
      f"{alpha_no_ctrl - alpha_true:>+10.3f}")
print(f"{'Naive Lasso':<25} {alpha_naive:>10.3f} "
      f"{alpha_naive - alpha_true:>+10.3f}")
print(f"{'Post-double selection':<25} {alpha_pds:>10.3f} "
      f"{alpha_pds - alpha_true:>+10.3f}")
print("=" * 55)


**Discussion:**
1. Why is the "no controls" estimate biased? In which direction?
2. Why might naive Lasso also give a biased estimate of $\alpha$?
3. Did $S_1$ and $S_2$ select different variables? Why is the **union** important?
4. How does this relate to BCH's gun law application from the lecture?

### Wrap-Up Questions

- What would happen to OLS, Ridge, and Lasso if we increased features to 200?
- What if we decreased the noise in the data?
- How many features did Lasso select, and does it match the true DGP?